# Aula 10 - Checkpoint: Contexto Longo

## Sua missao
Implemente `reduzir_contexto(documentos, pergunta, orcamento)`. A funcao deve reduzir o dossie do **Projeto Aurora**, preservar os cinco fatos necessarios e permitir que um LLM responda corretamente.

Esta e a unica atividade obrigatoria do checkpoint.

## Fluxo obrigatorio
1. Execute o cenario Aurora e observe a falha por excesso de contexto.
2. Implemente `reduzir_contexto`.
3. Valide a reducao e a resposta do LLM no cenario Aurora.
4. Reutilize a mesma estrategia no exemplo real LongBench.
5. Valide a resposta do LLM contra a referencia LongBench.
6. Preencha a conclusao final do notebook.

Nao aumente limites, nao use `FATOS_CRITICOS` dentro da funcao e nao use `answers` para selecionar o contexto LongBench.

In [9]:
from pathlib import Path

ORCAMENTO_TOKENS = 6_000       # Toda a entrada enviada ao OpenRouter.
ORCAMENTO_CONTEXTO = 5_000     # Reserva espaco para system prompt e pergunta.
MAX_TOKENS_SAIDA = 400         # Limita somente a resposta gerada.
PERGUNTA = 'Qual foi o incidente Aurora, em qual regiao ocorreu, qual versao resolveu, quem e o responsavel e qual o prazo do relatorio?'

FATOS_CRITICOS = [
    'INCIDENTE: INC-7842',
    'REGIAO: sa-east-1',
    'VERSAO DE ROLLBACK: 4.17.3',
    'RESPONSAVEL: Marina Costa',
    'PRAZO DO RELATORIO: 18/09/2026 17:00 BRT',
]

CAMINHO_DOSSIE = Path('dados/dossie_aurora.txt')
assert CAMINHO_DOSSIE.exists(), 'Dossie ausente. Execute gerar_dossie.py.'
contexto_bruto = CAMINHO_DOSSIE.read_text(encoding='utf-8')
documentos = contexto_bruto.split('\n\n--- FIM DO DOCUMENTO ---\n\n')
print(f'Arquivo: {CAMINHO_DOSSIE}')
print(f'Documentos: {len(documentos)}')
print(f'Caracteres: {len(contexto_bruto):,}')

Arquivo: dados\dossie_aurora.txt
Documentos: 260
Caracteres: 169,721


In [10]:
def estimar_tokens(texto):
    # Estimativa didatica. Tokenizadores reais variam por modelo.
    return max(1, len(texto) // 4)

tokens_brutos = estimar_tokens(contexto_bruto)
print(f'Tokens estimados: {tokens_brutos:,}')
print(f'Orcamento: {ORCAMENTO_TOKENS:,}')
print(f'Excesso: {tokens_brutos / ORCAMENTO_TOKENS:.1f}x')
assert tokens_brutos > ORCAMENTO_TOKENS, 'O cenario deveria exceder o orcamento.'

Tokens estimados: 42,430
Orcamento: 6,000
Excesso: 7.1x


In [11]:
class ContextoExcedido(Exception):
    pass

def validar_orcamento(texto, limite=ORCAMENTO_TOKENS):
    total = estimar_tokens(texto)
    if total > limite:
        raise ContextoExcedido(f'{total:,} tokens estimados para um limite de {limite:,}.')
    return total

# Esta falha e intencional: demonstra que concatenar tudo nao e uma solucao.
try:
    validar_orcamento(contexto_bruto)
except ContextoExcedido as erro:
    print('Falha esperada:', erro)

Falha esperada: 42,430 tokens estimados para um limite de 6,000.


## TODO 1 - Implemente a reducao
Complete apenas a funcao abaixo. Ela recebe a lista de documentos, a pergunta e o orcamento. Voce pode usar selecao por relevancia, deduplicacao, resumo ou uma combinacao. Nao use `FATOS_CRITICOS` para escolher documentos.

In [12]:
def reduzir_contexto(documentos, pergunta, orcamento):
    # TODO: selecione e/ou resuma documentos relevantes.
    # A saida deve ser texto, ficar abaixo de `orcamento` e preservar os fatos.
    import math
    import re
    import unicodedata
    from collections import Counter

    stopwords = {
        'a', 'o', 'e', 'de', 'do', 'da', 'dos', 'das', 'em', 'no', 'na', 'que', 'qual', 'quais', 'quem',
        'foi', 'um', 'uma', 'para', 'com', 'por', 'se', 'os', 'as', 'ao',
        'the', 'of', 'and', 'or', 'to', 'in', 'on', 'which', 'what', 'who', 'was', 'is', 'an', 'v', 'by', 'for',
    }
    abreviacao = re.compile(r'\b(?:v|vs|[A-Z]|U\.S|No|St|Mr|Mrs|Dr|Inc|Co|Ltd|Jr|Cir|Wis|Del|Wall|al)\.$')

    def normalizar_termos(texto):
        texto = unicodedata.normalize('NFKD', texto.lower()).encode('ascii', 'ignore').decode()
        return [t for t in re.findall(r'[a-z0-9]+(?:[.-][a-z0-9]+)*', texto) if t not in stopwords]

    def dividir_sentencas(linha):
        sentencas = []
        for parte in re.split(r'(?<=[.!?])\s+', linha):
            if sentencas and abreviacao.search(sentencas[-1]):
                sentencas[-1] += ' ' + parte  # Nao quebra em "Gates v. Collier" ou "413 U.S. 15".
            else:
                sentencas.append(parte)
        return sentencas

    # 1) Quebra cada documento em sentencas rotuladas pela origem e remove repeticoes.
    unidades, vistos = [], set()
    for i, doc in enumerate(documentos):
        linhas = [l.strip() for l in doc.strip().splitlines() if l.strip()]
        if not linhas:
            continue
        rotulo = linhas[0].split(' | ')[0][:80]
        if rotulo.endswith(':') and len(linhas) > 1:  # Ex.: 'Passage 3:' seguido do titulo.
            rotulo = f'{rotulo} {linhas[1][:80]}'
            linhas = linhas[2:]
        elif linhas[0].startswith(rotulo + ' | '):  # Ex.: 'DOC-017 | ...'.
            linhas[0] = linhas[0][len(rotulo) + 3:]
        posicao = 0
        for linha in linhas:
            for sentenca in dividir_sentencas(linha):
                chave = ' '.join(normalizar_termos(sentenca))
                if not chave or chave in vistos:
                    continue
                vistos.add(chave)
                unidades.append({'doc': i, 'pos': posicao, 'rotulo': rotulo,
                                 'texto': sentenca, 'termos': set(chave.split())})
                posicao += 1

    # 2) Pontua cada sentenca pelos termos da pergunta, ponderados por IDF (termos raros valem mais).
    consulta = set(normalizar_termos(pergunta))
    df = Counter(t for u in unidades for t in u['termos'] & consulta)
    idf = {t: math.log(1 + len(unidades) / df[t]) for t in df}
    relevancia_doc = Counter()
    for u in unidades:
        u['score'] = sum(idf[t] for t in u['termos'] & consulta)
        relevancia_doc[u['doc']] = max(relevancia_doc[u['doc']], u['score'])
    for u in unidades:
        # A sentenca de abertura de um documento relevante costuma situar o fato (data, tribunal, sistema).
        if u['score'] > 0 or u['pos'] == 0:
            u['score'] += 0.5 * relevancia_doc[u['doc']] / (1 + u['pos'])

    # 3) Seleciona as melhores sentencas ate o orcamento e restaura a ordem original.
    formatar = lambda u: f"[{u['rotulo']}] {u['texto']}"
    limite_caracteres = orcamento * 4  # Inverso de `estimar_tokens` (4 caracteres por token).
    corte = 0.4 * max((u['score'] for u in unidades), default=0)  # O orcamento e um teto, nao uma meta.
    candidatas = sorted((u for u in unidades if u['score'] > 0 and u['score'] >= corte),
                        key=lambda u: (-u['score'], u['doc'], u['pos']))
    escolhidas, usado = [], 0
    for u in candidatas:
        custo = len(formatar(u)) + 1  # +1 pela quebra de linha.
        if usado + custo <= limite_caracteres:
            escolhidas.append(u)
            usado += custo
    escolhidas.sort(key=lambda u: (u['doc'], u['pos']))
    return '\n'.join(formatar(u) for u in escolhidas)

# Use 5.000, nao 6.000: a requisicao ainda precisa comportar instrucoes e pergunta.
contexto_reduzido = reduzir_contexto(documentos, PERGUNTA, ORCAMENTO_CONTEXTO)
print('Tokens apos reducao:', validar_orcamento(contexto_reduzido, ORCAMENTO_CONTEXTO))

Tokens apos reducao: 78


In [13]:
def avaliar_solucao(contexto_reduzido):
    tokens = validar_orcamento(contexto_reduzido, ORCAMENTO_CONTEXTO)
    encontrados = {fato: fato in contexto_reduzido for fato in FATOS_CRITICOS}
    cobertura = sum(encontrados.values()) / len(FATOS_CRITICOS)
    reducao = 1 - tokens / tokens_brutos
    print(f'Tokens finais: {tokens:,}')
    print(f'Reducao: {reducao:.1%}')
    print(f'Cobertura: {cobertura:.0%}')
    for fato, ok in encontrados.items():
        print('[OK]' if ok else '[FALTOU]', fato)
    assert cobertura == 1, 'A reducao perdeu fatos criticos.'
    return {'tokens': tokens, 'reducao': reducao, 'cobertura': cobertura}

metricas = avaliar_solucao(contexto_reduzido)

Tokens finais: 78
Reducao: 99.8%
Cobertura: 100%
[OK] INCIDENTE: INC-7842
[OK] REGIAO: sa-east-1
[OK] VERSAO DE ROLLBACK: 4.17.3
[OK] RESPONSAVEL: Marina Costa
[OK] PRAZO DO RELATORIO: 18/09/2026 17:00 BRT


## TODO 2 - Consulte o LLM
Descomente as duas ultimas linhas da proxima celula somente depois que `avaliar_solucao` passar. A chamada deve usar exclusivamente `contexto_reduzido`.

In [14]:
import os
import time
import requests

ENDPOINT = 'https://openrouter.ai/api/v1/chat/completions'
MODELO = 'nvidia/nemotron-3-super-120b-a12b:free'  # modelo original falhava sempre
SYSTEM_PROMPT = 'Responda apenas com fatos sustentados pelo contexto. Nao invente. Preserve identificadores, nomes, datas e horarios exatamente como aparecem.'

# Carrega a chave do `.env` local, sem depender de python-dotenv
if Path('.env').exists():
    for linha in Path('.env').read_text(encoding='utf-8').splitlines():
        nome, _, valor = linha.partition('=')
        if nome.strip() and not nome.strip().startswith('#') and valor.strip():
            os.environ.setdefault(nome.strip(), valor.strip().strip('"\''))

def chamar_openrouter(contexto, pergunta, raciocinio=False, modelo=MODELO):
    mensagens = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'CONTEXTO:\n{contexto}\n\nPERGUNTA:\n{pergunta}'},
    ]
    tokens_entrada = validar_orcamento('\n'.join(m['content'] for m in mensagens), ORCAMENTO_TOKENS)
    print(f'Entrada enviada: {tokens_entrada:,}/{ORCAMENTO_TOKENS:,} tokens estimados')
    corpo = {
        'model': modelo,
        'messages': mensagens,
        'max_tokens': MAX_TOKENS_SAIDA,
        # Sem raciocinio, os 400 tokens de saida ficam inteiros para a resposta
        # Com raciocinio, a resposta voltava None porque o modelo gastava os 400 tokens de saída pensando
        'reasoning': {'enabled': raciocinio},
        # Impede o OpenRouter de comprimir o contexto silenciosamente.
        'plugins': [{'id': 'context-compression', 'enabled': False}],
    }

    api_key = os.getenv('OPENROUTER_API_KEY')
    assert api_key, 'Defina OPENROUTER_API_KEY no ambiente ou no arquivo .env.'
    inicio = time.perf_counter()
    resposta = requests.post(
        ENDPOINT,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json=corpo,
        timeout=90,
    )
    if resposta.status_code == 401:
        raise RuntimeError('Chave OpenRouter ausente, invalida ou expirada.')
    if resposta.status_code == 429:
        raise RuntimeError('Rate limit: escolha outro modelo gratuito disponivel.')
    resposta.raise_for_status()
    dados = resposta.json()
    uso = dados.get('usage') or {}
    return {
        'latencia_s': round(time.perf_counter() - inicio, 1),
        'finish_reason': dados['choices'][0].get('finish_reason'),
        'tokens_saida': uso.get('completion_tokens'),
        'tokens_raciocinio': (uso.get('completion_tokens_details') or {}).get('reasoning_tokens'),
        'resposta': dados['choices'][0]['message'].get('content') or '',
    }

def consultar_llm(contexto, pergunta, modelo=MODELO):
    resposta = chamar_openrouter(contexto, pergunta, modelo=modelo)['resposta']
    assert resposta.strip(), 'O modelo devolveu resposta vazia.'
    return resposta

# Execute somente depois de implementar e validar `contexto_reduzido`.
resposta_llm = consultar_llm(contexto_reduzido, PERGUNTA)
print(resposta_llm)

Entrada enviada: 150/6,000 tokens estimados
INCIDENTE: INC-7842  
REGIÃO: sa-east-1  
VERSÃO DE ROLLBACK: 4.17.3  
RESPONSÁVEL: Marina Costa  
PRAZO DO RELATÓRIO: 18/09/2026 17:00 BRT


In [15]:
def avaliar_resposta_llm(resposta):
    texto = resposta.lower()
    criterios = {
        'incidente': 'inc-7842' in texto,
        'regiao': 'sa-east-1' in texto,
        'versao': '4.17.3' in texto,
        'responsavel': 'marina costa' in texto,
        'prazo': all(valor in texto for valor in ['18/09/2026', '17:00', 'brt']),
    }
    for criterio, ok in criterios.items():
        print('[OK]' if ok else '[FALTOU]', criterio)
    assert all(criterios.values()), 'A resposta do LLM perdeu ou alterou fatos criticos.'
    return criterios

avaliacao_llm = avaliar_resposta_llm(resposta_llm)

[OK] incidente
[OK] regiao
[OK] versao
[OK] responsavel
[OK] prazo


## Entrega
- Notebook executado com a falha inicial, reducao e resposta real do LLM.
- Contexto final abaixo de 6.000 tokens estimados e cobertura local de 100% dos fatos.
- Resposta do LLM aprovada por `avaliar_resposta_llm`.
- Chat Python ou WPF aplicando a politica escolhida.
- README explicando estrategia, perdas possiveis, custo, latencia e efeito do cache KV em VRAM.

## TODO 3 - Aplique no LongBench
Esta etapa e obrigatoria. Reutilize `reduzir_contexto` no contexto real abaixo. O campo `answers` serve somente para avaliacao depois da resposta; nao o use para recuperar ou resumir passagens.

In [16]:
import json
import re

CAMINHO_LONGBENCH = Path('dados/longbench_hotpotqa_exemplo.json')
exemplo_real = json.loads(CAMINHO_LONGBENCH.read_text(encoding='utf-8'))
documentos_longbench = [p for p in re.split(r'(?=Passage \d+:\n)', exemplo_real['context']) if p.strip()]
pergunta_longbench = exemplo_real['input']
print('Fonte:', exemplo_real['source'])
print('Pergunta:', pergunta_longbench)
print('Passagens:', len(documentos_longbench))
print('Tokens estimados:', f"{estimar_tokens(exemplo_real['context']):,}")

# TODO: use a mesma funcao criada no cenario Aurora.
contexto_longbench = reduzir_contexto(documentos_longbench, pergunta_longbench, ORCAMENTO_CONTEXTO)
print('Tokens apos reducao:', validar_orcamento(contexto_longbench, ORCAMENTO_CONTEXTO))
# Instrução oficial do LongBench/hotpotqa: resposta curta, sem justificativa.
pergunta_longbench_prompt = (f'{pergunta_longbench}\n'
                             'Answer the question based on the given passages. '
                             'Only give me the answer and do not output any other words.')
resposta_longbench = consultar_llm(contexto_longbench, pergunta_longbench_prompt)
print(resposta_longbench)

Fonte: zai-org/LongBench - hotpotqa - primeiro exemplo
Pergunta: Which case was brought to court first Miller v. California or Gates v. Collier ?
Passagens: 10
Tokens estimados: 13,515
Tokens apos reducao: 1565
Entrada enviada: 1,653/6,000 tokens estimados
Miller v. California was brought to court first.


In [17]:
def avaliar_longbench(resposta, respostas_referencia):
    texto = resposta.lower()
    correspondencias = {ref: ref.lower() in texto for ref in respostas_referencia}
    for referencia, ok in correspondencias.items():
        print('[OK]' if ok else '[FALTOU]', referencia)
    assert any(correspondencias.values()), 'A resposta nao corresponde a referencia LongBench.'
    return correspondencias

avaliacao_longbench = avaliar_longbench(resposta_longbench, exemplo_real['answers'])

[OK] Miller v. California


## COM raciocinio X SEM raciocinio

In [18]:
comparacao = {}
for raciocinio in (True, False):
    r = chamar_openrouter(contexto_longbench, pergunta_longbench_prompt, raciocinio=raciocinio)
    # `answers` usado somente para avaliar, depois da resposta.
    r['aprovada'] = any(ref.lower() in r['resposta'].lower() for ref in exemplo_real['answers'])
    comparacao['COM raciocinio' if raciocinio else 'SEM raciocinio'] = r

for nome, r in comparacao.items():
    print(f'=== {nome}')
    print(f"Latencia: {r['latencia_s']}s | finish_reason: {r['finish_reason']} | "
          f"tokens de saida: {r['tokens_saida']} (raciocinio: {r['tokens_raciocinio']}) | aprovada: {r['aprovada']}")
    print('Resposta:', r['resposta'].strip() or '(vazia - limite de saida consumido pelo raciocinio)')

Entrada enviada: 1,653/6,000 tokens estimados
Entrada enviada: 1,653/6,000 tokens estimados
=== COM raciocinio
Latencia: 3.1s | finish_reason: stop | tokens de saida: 356 (raciocinio: 348) | aprovada: False
Resposta: Gates v. Collier
=== SEM raciocinio
Latencia: 1.3s | finish_reason: stop | tokens de saida: 5 (raciocinio: 0) | aprovada: True
Resposta: Miller v. California


## Análise

- Por que COM raciocinio não foi aprovada e SEM raciocinio foi?<br>
Com raciocinio, o modelo considerou o início do processo (Gates v. Collier, 1970-1971) e respondeu Gates. Mas, a referência do LongBench usa o ano da decisão (Miller 1973 x Gates 1974) e o SEM raciocinio foi direto nesses anos, respondendo corretamente Miller v. California

- Diferença entre os dois:
    - COM raciocinio: 3,1 segundos e 356 tokens de saida (348 só de raciocínio)
    - SEM raciocinio: 1,3 segundos e 5 tokens

- Qual é melhor nesse caso? <br>
Nesse caso, o SEM raciocínio acertou a referência, foi ~2,4x mais rapido e usou ~71x menos tokens. Como o contexto reduzido já traz os fatos, raciocinar não foi benéfico e ainda arriscou respostas vazias com o limite de 400 tokens

## TODO 4 - Conclusao e entrega
- Qual estratégia foi usada nos dois cenários?<br>
Redução extrativa, dividindo os documentos em frases, removendo repetições e mantendo as frases com mais termos da pergunta (termos raros valem mais), até o orçamento

- Quantos tokens havia antes e depois em cada um?<br>
    - Aurora: 42.430 → 78 (99,8% menor)
    - LongBench: 13.515 → 1.565 (88,4% menor)

- A mesma estratégia funcionou igualmente no dado sintético e no real?<br>
Não. No Aurora, cada fato tem termos exclusivos da pergunta e o LLM acertou tudo. No LongBench, os termos são comuns, entram passagens que só atrapalham e a resposta só bateu com a referência sem raciocínio

- O que poderia ser perdido pela estratégia?<br>
Fatos escritos sem os termos da pergunta (sinônimos, pronomes, datas em outra frase) e a ligação entre frases

- Como contexto longo afeta custo, latência e cache KV/VRAM?<br>
Mais tokens de entrada aumentam o custo e a latência. O cache KV cresce a cada token, ocupando mais VRAM e reduzindo quantas requisições rodam ao mesmo tempo